# 1. Imports and configuration

In [ ]:
from pathlib import Path

import numpy as np
import torch
from PIL import Image

from transformers import pipeline, CLIPModel, CLIPProcessor

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs/semantic")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

# 2. Semantic classes, prompts, and costs

In [18]:
CLASSES = [
    "traversable dirt or gravel ground",
    "traversable grass",
    "dense vegetation",
    "solid obstacle",
    "muddy terrain",
    "water",
    "sky",
    "vehicle"
]

PROMPTS = [
    "an off-road surface consisting of clear traversable dirt or gravel",
    "an area of short grass that an off-road vehicle could drive over",
    "dense bushes, shrubs, or vegetation that blocks a vehicle",
    "a solid obstacle such as a rock, boulder, tree trunk, or barrier",
    "muddy off-road terrain",
    "water or a puddle",
    "the sky",
    "a vehicle"
]

CLASS_COSTS = {
    "traversable dirt or gravel ground": 0.0,
    "traversable grass": 0.25,
    "dense vegetation": 0.9,
    "solid obstacle": 1.0,
    "muddy terrain": 0.6,
    "water": 0.9,
    "sky": 1.0,
    "vehicle": 1.0
}

# 3. Load SAM 2 and CLIP

In [ ]:
print("Loading SAM 2...")

mask_generator = pipeline(
    task="mask-generation",
    model="facebook/sam2.1-hiera-small",
    device=0 if DEVICE == "cuda" else -1
)

print("Loading CLIP...")

clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(DEVICE)

clip_model.eval()

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

print("Models loaded.")

# 4. Helper functions

In [ ]:
# Convert the image to RGB while preserving its original resolution.
# (With some modifications, it can also be used for image resizing.)
def prepare_image(image):
    return image.convert("RGB")


def make_masked_crop(image_np, mask_info, padding=0.08):
    mask = mask_info["mask"]

    ys, xs = np.where(mask)

    if len(xs) == 0:
        return None

    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()

    width = x2 - x1 + 1
    height = y2 - y1 + 1

    pad_x = int(width * padding)
    pad_y = int(height * padding)

    x1 = max(0, x1 - pad_x)
    x2 = min(image_np.shape[1], x2 + pad_x + 1)

    y1 = max(0, y1 - pad_y)
    y2 = min(image_np.shape[0], y2 + pad_y + 1)

    crop = image_np[y1:y2, x1:x2].copy()
    crop_mask = mask[y1:y2, x1:x2]

    # Neutral background outside the actual SAM region
    masked_crop = np.full_like(crop, 127)

    # Keep only pixels belonging to the SAM mask
    masked_crop[crop_mask] = crop[crop_mask]

    return Image.fromarray(masked_crop)

# 5. Complete semantic pipeline

In [21]:
def get_semantic_cost_map(image):
    """
    Generate a semantic traversability cost map.

    Returns
    -------
    np.ndarray
        Shape: same height and width as the input image
        dtype: float32

        0.0 = traversable
        1.0 = non-traversable
    """

    # --------------------------------------------------
    # 1. Preprocess image
    # --------------------------------------------------

    image = prepare_image(image)

    image_np = np.asarray(image)

    H, W = image_np.shape[:2]
    image_area = H * W

    # --------------------------------------------------
    # 2. SAM 2 automatic segmentation
    # --------------------------------------------------

    sam_outputs = mask_generator(
        image,
        points_per_batch=64,
        pred_iou_thresh=0.70,
        stability_score_thresh=0.80
    )

    mask_infos = []

    for i, mask in enumerate(sam_outputs["masks"]):
        if torch.is_tensor(mask):
            mask_np = mask.cpu().numpy()
        else:
            mask_np = np.asarray(mask)

        mask_np = mask_np.astype(bool)

        area = int(mask_np.sum())
        frac = area / image_area

        if 0.001 <= frac <= 0.90:
            mask_infos.append({
                "index": i,
                "mask": mask_np,
                "area": area,
                "frac": frac
            })

    # Keep at most 25 largest useful regions
    mask_infos = sorted(
        mask_infos,
        key=lambda x: x["area"],
        reverse=True
    )[:25]

    # --------------------------------------------------
    # 3. Make CLIP crops
    # --------------------------------------------------

    crops = [
        make_masked_crop(image_np, mask_info)
        for mask_info in mask_infos
    ]

    # Default value for pixels not covered by retained masks
    semantic_cost_map = np.full(
        (H, W),
        0.5,
        dtype=np.float32
    )

    if len(crops) == 0:
        return semantic_cost_map

    # --------------------------------------------------
    # 4. CLIP classification
    # --------------------------------------------------

    inputs = clip_processor(
        text=PROMPTS,
        images=crops,
        return_tensors="pt",
        padding=True
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = clip_model(**inputs)

    probabilities = (
        outputs.logits_per_image
        .softmax(dim=1)
        .cpu()
        .numpy()
    )

    # --------------------------------------------------
    # 5. Convert semantic probabilities to costs
    # --------------------------------------------------

    results = []

    for mask_info, probs in zip(
        mask_infos,
        probabilities
    ):
        cost = sum(
            float(probs[i]) * CLASS_COSTS[CLASSES[i]]
            for i in range(len(CLASSES))
        )

        results.append({
            "mask": mask_info["mask"],
            "area": mask_info["area"],
            "cost": cost
        })

    # Large general regions first.
    # Smaller regions overwrite them.
    results = sorted(
        results,
        key=lambda x: x["area"],
        reverse=True
    )

    for result in results:
        semantic_cost_map[
            result["mask"]
        ] = result["cost"]

    # --------------------------------------------------
    # 6. Guarantee output format
    # --------------------------------------------------

    semantic_cost_map = np.clip(
        semantic_cost_map,
        0.0,
        1.0
    ).astype(np.float32)

    return semantic_cost_map

# 6. Process the whole directory

In [ ]:
IMAGE_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg"
}

image_files = sorted([
    path
    for path in DATA_DIR.iterdir()
    if (
        path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )
])

print(f"Found {len(image_files)} frames.")

In [ ]:
for i, image_path in enumerate(image_files, start=1):

    output_path = (
        OUTPUT_DIR
        / f"{image_path.stem}_semantic_cost.npy"
    )

    # Useful if processing is interrupted:
    # don't redo completed frames.
    if output_path.exists():
        print(
            f"[{i}/{len(image_files)}] "
            f"Skipping {image_path.name}"
        )
        continue

    print(
        f"[{i}/{len(image_files)}] "
        f"Processing {image_path.name}"
    )

    try:
        image = Image.open(image_path)

        cost_map = get_semantic_cost_map(image)

        np.save(
            output_path,
            cost_map
        )

    except Exception as e:
        print(f"    ERROR: {e}")

# 7. Validation

Sanity check: verify that every saved .npy file has the right shape, float32 dtype, values in [0,1], and no NaN/Inf values.

In [ ]:
npy_files = sorted(
    OUTPUT_DIR.glob("*_semantic_cost.npy")
)

print("Number of .npy files:", len(npy_files))

# Use the input image dimensions as the expected output dimensions.
with Image.open(image_files[0]) as img:
    expected_shape = (img.height, img.width)

bad_files = []

for path in npy_files:
    cost_map = np.load(path)

    problems = []

    if cost_map.shape != expected_shape:
        problems.append(
            f"shape={cost_map.shape}, expected={expected_shape}"
        )

    if cost_map.dtype != np.float32:
        problems.append(f"dtype={cost_map.dtype}")

    if cost_map.min() < 0 or cost_map.max() > 1:
        problems.append(
            f"range=[{cost_map.min()}, {cost_map.max()}]"
        )

    if not np.isfinite(cost_map).all():
        problems.append("contains NaN/Inf")

    if problems:
        bad_files.append((path.name, problems))

print("Valid files:", len(npy_files) - len(bad_files))
print("Problem files:", len(bad_files))

for name, problems in bad_files[:20]:
    print(name, problems)

Information for one sample map.

In [ ]:
sample_path = npy_files[0]
cost_map = np.load(sample_path)

print("File:", sample_path.name)
print("Shape:", cost_map.shape)
print("dtype:", cost_map.dtype)
print("Min:", cost_map.min())
print("Max:", cost_map.max())
print("Mean:", cost_map.mean())

Visual sanity check of that same sample map.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 7))

plt.imshow(
    cost_map,
    cmap="magma",
    vmin=0,
    vmax=1
)

plt.colorbar(
    label="Semantic traversability cost"
)

plt.title(sample_path.name)
plt.axis("off")
plt.show()